# 24 · Set 7c: can an answer gate keep the new pipeline's accuracy at lower compute? (A100, about 1–1.5 hours)

**The question:** with the fine-tuned Qwen 3B retriever (notebook 23a), Qwen 32B answering every question reaches 84.2% (corrected
label) / 64.0% (exact match). Can Llama 8B answer most questions and hand only its least-sure answers to Qwen 32B, keeping most of
that accuracy while saving more compute?

**Why this needs the GPU:** the probe must read the hidden state of the exact Llama 8B answer it judges. The new retriever changes
the passages, hence Llama's answers and hidden states. So Llama 8B answers again on the new shortlists, and the hidden states of
that same pass are saved (same code as notebooks 16–17).

| | |
|---|---|
| Shortlists | The fine-tuned 3B retriever's (notebook 23a), unchanged |
| Llama 8B | One greedy step-by-step answer (same prompt as before), hidden states saved from that same pass |
| Qwen 32B | Its answers on the same shortlists already exist (notebook 23a) |
| Fitted inside training folds | The probe (layer 28 of 32, final answer line, 16 components, logistic regression), its target ("Llama's answer is no worse than Qwen's"), and the share escalated |
| Policy named beforehand | The smallest escalation share whose training accuracy is within 1 point of always-Qwen; also fixed shares of 30%, 40% and 50% |
| Judged against | Always Qwen 32B on the new shortlists (the improved pipeline), the cheap fixed setup (AgentTTS's pick), Llama 8B alone, and a random gate with the same share |
| Evaluation | All 400 questions, 5-fold cross-validation repeated 5 times; and old → fresh, fresh → old |

**My estimate:** Llama 8B copy about 5–8 minutes, answers with hidden states about 45–75 minutes, analysis about 2 minutes.
If the session stops, run the notebook again: it continues.

Output: `runs/Set_3__self-learnt-retrieval/<timestamp>_<GPU>_gate3_12k/results/` (`summary.md`, `set7c_answer_gate.xlsx`, `charts/gate_curve.png`).

**Runtime → Change runtime type → A100 GPU.**

In [ ]:
# ==== settings ====
PROJECT_DIR = "/content/drive/MyDrive/0_EngD_assignments/0_Small_Big_Model_Switching"
EXPERIMENT  = "Set_3__self-learnt-retrieval"
EXP_SET5    = "Set_2__question-reading-routers"
OLD_SWEEP_EXPERIMENT = "Set_1__3-modality-size"
FRESH_DATA  = f"{PROJECT_DIR}/data/hotpotqa_q/fresh200_c20_s43"
RETRIEVER_RUN_TAG = "_ft3_12k"        # the fine-tuned retriever whose shortlists are used (notebook 23a)
CV_REPEATS = 5
TIME_LIMIT_MIN = 150                  # stops cleanly at this limit; run the notebook again to continue
NEW_RUN = False                       # False = continue the latest run of this notebook
DRY_RUN = False                       # True = tiny model on CPU: tests the whole notebook in a few minutes

EXPECTED_CODE = {
 "ssw.py": "c9fbdc7f6b91c58b607021b8be8c03ae2ff67dc1024fd2e24fffeb18e239ed98",
 "sweep.py": "9382292c6fe396cf79c73b212aaf6c2d677cae7f9311c3a5304f0578b52e6bb6",
 "routers.py": "8a293df1c8955716950f535cf9a3ced24439e00250173da1a3df110c0ed87304",
 "hidden.py": "574e6c1390e47c6de833a0c10b32c24eed0dee73a35eabc17f58842835e7f105",
 "final.py": "f18a6fba3d1bdd1f6700f6dc535f10ec0c41dbf5192b7b39ad097f8a1b88be53",
 "retriever_check.py": "fde08c549ecbd6bfc5527a42c731aa51dcff687a64818d68cf9cb30090e017e3",
 "ensemble.py": "5c5e87175d2484d806fda913a9555f71378a6031f050c37e89c20305942b76fc",
 "experts.py": "17ed39957e54853d178b47fa73ba328b50f0589b129ccb191a2efb0a2dea466b",
 "retft.py": "4a0d27d24459ba179c60dcce39a85285ae422dfa9e8d7cbc160e79a0a9932879",
 "gate7.py": "e904bba28a163c53dbf23c7ce510cd316223cc18a5a7bbc5c5b2a1dade8d1d08"
}

In [ ]:
import subprocess, sys, os, json, shutil, time
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader 2>/dev/null",
                     shell=True, capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "none")
if not DRY_RUN:
    assert gpu, "No GPU: Runtime -> Change runtime type -> A100 GPU (or set DRY_RUN = True)."
    if "T4" in gpu:
        raise SystemExit("This runtime has a T4, which cannot run this: the 7B model alone (15 GB) does not fit in its memory.\n"
                         "Runtime -> Change runtime type -> A100 GPU (or G4 / L4) -> Save, then run all cells again.")
    if "A100" not in gpu and "G4" not in gpu and "RTX PRO 6000" not in gpu:
        print("Note: planned for an A100 or G4; this GPU works but timings will differ.")
GPU_TAG = "cpu" if not gpu else gpu.split(",")[0].replace("NVIDIA", "").replace("Tesla", "").strip().split()[0].split("-")[0]
T_START = time.time()

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

### Check that Drive is serving the current code; find the earlier runs

In [ ]:
import glob, hashlib
from datetime import datetime
from zoneinfo import ZoneInfo
def sha(p):
    return hashlib.sha256(open(p, "rb").read()).hexdigest()
CODE_DIR = "/content/set7code"
os.makedirs(CODE_DIR, exist_ok=True)
stale = []
for f, want in EXPECTED_CODE.items():
    shutil.copy(f"{PROJECT_DIR}/code/{f}", f"{CODE_DIR}/{f}")
    if sha(f"{CODE_DIR}/{f}") != want:
        stale.append(f)
assert not stale, (f"Drive is serving a different version of {stale} than this notebook was made for. Wait two minutes, then "
                   "Runtime -> Disconnect and delete runtime, and run again.")
sweeps = [d for d in sorted(glob.glob(f"{PROJECT_DIR}/runs/{OLD_SWEEP_EXPERIMENT}/*")) if "_dryrun" not in d and os.path.exists(f"{d}/results/per_question.jsonl")
          and os.path.exists(f"{d}/sweep_status.json") and json.load(open(f"{d}/sweep_status.json")).get("complete")]
fresh = [d for d in sorted(glob.glob(f"{PROJECT_DIR}/runs/{EXP_SET5}/*_final")) if os.path.exists(f"{d}/step/retrieval_q7_c1_p3.jsonl")]
ft = [d for d in sorted(glob.glob(f"{PROJECT_DIR}/runs/{EXPERIMENT}/*{RETRIEVER_RUN_TAG}")) if os.path.exists(f"{d}/step/answers_q32.jsonl")]
assert sweeps and fresh and ft, "Needs the finished runs of notebooks 11, 17 and 23a."
OLD_SWEEP, FRESH_RUN, FT_RUN = sweeps[-1], fresh[-1], ft[-1]
RET_FILE = glob.glob(f"{FT_RUN}/step/retrieval_ft*.jsonl")[0]
OTHER_ANSWERS = [d for d in sorted(glob.glob(f"{PROJECT_DIR}/runs/{EXPERIMENT}/*_ft*")) if d != FT_RUN and "_dryrun" not in d and os.path.exists(f"{d}/step/answers_q32.jsonl")]
OLD_DATA = json.load(open(f"{OLD_SWEEP}/config.json"))["data_dir"]
SESSION = datetime.now(ZoneInfo("Asia/Singapore")).strftime("%Y%m%d-%H%M%S")
tag = "_gate3_12k_dryrun" if DRY_RUN else "_gate3_12k"
prev = sorted(glob.glob(f"{PROJECT_DIR}/runs/{EXPERIMENT}/*{tag}"))
RUN_DIR = prev[-1] if (prev and not NEW_RUN) else f"{PROJECT_DIR}/runs/{EXPERIMENT}/{SESSION}_{GPU_TAG}{tag}"
SESSION_DIR = f"{RUN_DIR}/sessions/{SESSION}"
for d in (f"{RUN_DIR}/code", f"{RUN_DIR}/logs", f"{RUN_DIR}/step", SESSION_DIR):
    os.makedirs(d, exist_ok=True)
for f in EXPECTED_CODE:
    shutil.copy(f"{CODE_DIR}/{f}", f"{RUN_DIR}/code/{f}")
TIMES = {}
print("Code files: current.")
print("Fine-tuned retriever run:", FT_RUN)
print("Its shortlists:          ", RET_FILE)
print("Run folder:              ", RUN_DIR)

In [ ]:
MODELS = {   # key: model, folder under models/, parameters (for compute), nominal size (for the AgentTTS budget formula)
    "q3":  {"id": "Qwen/Qwen2.5-3B-Instruct",        "dir": "Qwen2.5-3B-Instruct",        "params": 3.09e9,  "nominal": 3e9},
    "q7":  {"id": "Qwen/Qwen2.5-7B-Instruct",        "dir": "Qwen2.5-7B-Instruct",        "params": 7.62e9,  "nominal": 7e9},
    "q32": {"id": "Qwen/Qwen2.5-32B-Instruct-AWQ",   "dir": "Qwen2.5-32B-Instruct-AWQ",   "params": 32.76e9, "nominal": 32e9},
    "l3":  {"id": "meta-llama/Llama-3.2-3B-Instruct", "dir": "Llama-3.2-3B-Instruct",     "params": 3.21e9,  "nominal": 3e9},
    "l8":  {"id": "meta-llama/Llama-3.1-8B-Instruct", "dir": "Llama-3.1-8B-Instruct",     "params": 8.03e9,  "nominal": 8e9},
}
LOCAL = "/content/models"
for m in MODELS.values():
    m["path"] = f"{LOCAL}/{m['dir']}"

MODELS = {k: v for k, v in MODELS.items() if k in ("q7", "l8", "q32")}
json.dump(MODELS, open(f"{RUN_DIR}/models.json", "w"), indent=1)
json.dump({"experiment": EXPERIMENT, "set": "7c answer gate on the fine-tuned 3B retriever", "retriever_run": FT_RUN, "ret_file": RET_FILE,
           "old_sweep": OLD_SWEEP, "fresh_run": FRESH_RUN, "old_data": OLD_DATA, "fresh_data": FRESH_DATA, "cv_repeats": CV_REPEATS,
           "gpu": gpu, "dry_run": DRY_RUN, "code": EXPECTED_CODE, "started": SESSION}, open(f"{RUN_DIR}/config.json", "w"), indent=1)

def minutes_left():
    return max(0.0, TIME_LIMIT_MIN - (time.time() - T_START) / 60)

def run_stage(args, log_name):
    cmd = [sys.executable, "-u", f"{CODE_DIR}/gate7.py"] + [str(a) for a in args]
    t0 = time.time()
    with open(f"{RUN_DIR}/logs/{log_name}", "a") as log, open(f"{SESSION_DIR}/{log_name}", "a") as slog:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            log.write(line); slog.write(line)
            if not any(k in line for k in ("it/s]", "s/it]", "Loading checkpoint", "Loading weights", "Warning", "warn")):
                print(line, end="")
        p.wait()
    TIMES[log_name.replace(".log", "")] = TIMES.get(log_name.replace(".log", ""), 0) + round(time.time() - t0)
    json.dump(TIMES, open(f"{SESSION_DIR}/stage_times.json", "w"), indent=1)
    print(f"--- {log_name}: exit {p.returncode}, {time.time() - t0:.0f}s, {minutes_left():.0f} min left of the limit")
    assert p.returncode == 0, f"stage failed - see {RUN_DIR}/logs/{log_name}"

COMMON = ["--old_data", OLD_DATA, "--fresh_data", FRESH_DATA, "--run_dir", RUN_DIR, "--models_json", f"{RUN_DIR}/models.json", "--ret_file", RET_FILE]
def n_done():
    p = f"{RUN_DIR}/step/hidden_answer_l8.jsonl"
    return sum(1 for _ in open(p)) if os.path.exists(p) else 0
print("Llama 8B answers with hidden states so far:", n_done(), "of 400")

### 1. Llama 8B answers on the new shortlists, with the hidden states of that same pass (resumable)

In [ ]:
if n_done() < 400:
    if DRY_RUN:
        run_stage(["extract", "--backend", "tiny", "--max_new", 12] + COMMON, "extract.log")
    else:
        m = MODELS["l8"]
        if not os.path.exists(f"{m['path']}/config.json"):
            t = time.time()
            print("copying Llama 8B to local disk ...")
            shutil.copytree(f"{PROJECT_DIR}/models/{m['dir']}", m["path"] + ".part", dirs_exist_ok=True)
            os.rename(m["path"] + ".part", m["path"])
            TIMES["copy_l8"] = round(time.time() - t)
        run_stage(["extract", "--time_limit_min", max(5.0, minutes_left() - 5)] + COMMON, "extract.log")
ALL_DONE = n_done() >= 400
print("\nAll 400 answers extracted." if ALL_DONE else "\nNOT COMPLETE YET - run this notebook again to continue.")

### 2. The gate, fitted inside training folds (CPU)

In [ ]:
if ALL_DONE:
    from IPython.display import Image, Markdown, display
    run_stage(["analyze", "--old_sweep", OLD_SWEEP, "--fresh_run", FRESH_RUN, "--answers_run", FT_RUN, "--extra_answers"] + OTHER_ANSWERS
              + ["--repeats", CV_REPEATS] + COMMON, "analyze.log")
    display(Markdown(open(f"{RUN_DIR}/results/summary.md").read()))
    display(Image(f"{RUN_DIR}/results/charts/gate_curve.png"))
    print("Excel workbook:", f"{RUN_DIR}/results/set7c_answer_gate.xlsx")
TIMES["total_session"] = round(time.time() - T_START)
json.dump({"session": SESSION, "run_dir": RUN_DIR, "gpu": gpu, "seconds": TIMES, "code": EXPECTED_CODE, "complete": ALL_DONE},
          open(f"{SESSION_DIR}/session.json", "w"), indent=1)
print("Wall-clock this session (s):", TIMES)
print("\nALL DONE." if ALL_DONE else "\nNOT COMPLETE YET - run this notebook again to continue.")
try:
    drive.flush_and_unmount()
    print("Saved to Drive.")
except Exception as e:
    print("Drive flush skipped:", e)